# 注意
下面代码增加了调用通义千问的，但是通义的不支持以下方式定义格式化返回，所以输出和视频讲解的不一样，第二个单元格用https://models.inference.ai.azure.com/的可以掉gpt模型，你可以分别执行下，看下返回内容
settings = OpenAIChatPromptExecutionSettings(response_format=TravelPlan)

In [7]:
import json  # 导入 Python 内置的 JSON 库，用于处理 JSON 数据的序列化和反序列化
import os    # 导入操作系统接口库，用于读取环境变量或处理文件路径

from dotenv import load_dotenv  # 从 python-dotenv 库导入，用于从本地 .env 文件中加载环境变量

from pydantic import BaseModel, ValidationError, Field  # 导入 Pydantic 组件，用于定义具有数据校验功能的模型和字段
from typing import List  # 导入类型提示组件，用于在代码中明确标注列表（List）类型的数据

from openai import AsyncOpenAI  # 导入 OpenAI 的异步客户端，用于直接与 OpenAI API 进行非阻塞通信

# 从 Semantic Kernel 的 OpenAI 适配器中导入聊天完成服务类和执行设置类
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings

# 导入 Semantic Kernel 的代理（Agent）核心类：ChatCompletionAgent（对话代理）和 ChatHistoryAgentThread（对话历史线程）
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread

from semantic_kernel.functions import KernelArguments  # 导入内核参数类，用于在调用插件函数时传递各种参数

In [8]:
load_dotenv()

model_name = "gpt-4.1-mini"
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# model_name = "qwen-max"
# client = AsyncOpenAI(
#     api_key=os.environ.get("DASHSCOPE_API_KEY"), 
#     base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
# )

chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,
    async_client=client,
)

In [10]:
# 定义子任务模型，继承自 Pydantic 的 BaseModel 以获得自动校验和序列化功能
class SubTask(BaseModel):
    # 定义负责该子任务的代理名称，使用 Field 添加描述供 LLM 理解其含义
    assigned_agent: str = Field(
        description="分配给负责处理此子任务的具体代理（Agent）名称")
    
    # 定义子任务的详细描述，明确告知 LLM 需要在这里填入具体的执行步骤
    task_details: str = Field(
        description="对该子任务需要完成的具体内容的详细描述")


# 定义整体旅行计划模型，作为顶层数据结构
class TravelPlan(BaseModel):
    # 存储用户最初提出的原始旅行需求或总目标
    main_task: str = Field(
        description="用户提出的整体旅行需求或总任务")
    
    # 定义一个子任务列表，类型为前面定义的 SubTask 对象的集合
    # 这种嵌套结构允许 LLM 将复杂任务拆解为多个由不同代理负责的原子任务
    subtasks: List[SubTask] = Field(
        description="从主任务拆解出来的子任务列表，每个子任务都分配给一个专门的代理")

In [12]:
AGENT_NAME = "TravelAgent"
"""
您是一名规划者智能体（Planner Agent）。
您的工作是根据用户的请求，决定应该运行哪些智能体。
以下是专精于不同任务的可用智能体：
- FlightBooking（航班预订）：用于预订航班和提供航班信息
- HotelBooking（酒店预订）：用于预订酒店和提供酒店信息
- CarRental（汽车租赁）：用于预订汽车和提供汽车租赁信息
- ActivitiesBooking（活动预订）：用于预订活动和提供活动信息
- DestinationInfo（目的地信息）：用于提供有关目的地的信息
- DefaultAgent（默认智能体）：用于处理一般性请求
"""
AGENT_INSTRUCTIONS = """您是一名规划者智能体（Planner Agent）。
您的工作是根据用户的请求，决定应该运行哪些智能体。
以下是专精于不同任务的可用智能体：
- FlightBooking（航班预订）：用于预订航班和提供航班信息
- HotelBooking（酒店预订）：用于预订酒店和提供酒店信息
- CarRental（汽车租赁）：用于预订汽车和提供汽车租赁信息
- ActivitiesBooking（活动预订）：用于预订活动和提供活动信息
- DestinationInfo（目的地信息）：用于提供有关目的地的信息
- DefaultAgent（默认智能体）：用于处理一般性请求"""

# 创建提示执行设置并配置Pydantic模型响应格式
# --- 1. 配置执行设置（执行环境的“潜规则”） ---
# 实例化 OpenAI 专用的执行设置对象。
# response_format=TravelPlan: 这是最关键的一步，利用 OpenAI 的“结构化输出”特性，
# 强制要求模型返回的结果必须符合 TravelPlan 类定义的 JSON 结构，否则会报错。
settings = OpenAIChatPromptExecutionSettings(response_format=TravelPlan)

# --- 2. 初始化智能代理（Agent） ---
# 创建一个对话型的智能代理实例。
agent = ChatCompletionAgent(
    service=chat_completion_service,  # 指定底层使用的 AI 服务（如配置好的 OpenAI 连接）
    name=AGENT_NAME,                  # 设置 Agent 的名称，用于在多代理系统中标识身份
    instructions=AGENT_INSTRUCTIONS,  # 注入系统提示词，定义 Agent 的角色、目标和行为准则
    
    # 将前面定义的执行设置包装在 KernelArguments 中传递给 Agent。
    # 这意味着该 Agent 之后所有的对话（Invoke）都会默认遵循这些设置（即必须返回 TravelPlan 格式）。
    arguments=KernelArguments(settings) 
)

In [13]:
from IPython.display import display, HTML  # 导入用于在 Notebook 中渲染富文本（HTML）的工具

async def main():
    # 初始化对话线程对象。
    # 如果是第一次对话，设为 None；后续 Agent 会返回更新后的 thread 以维持记忆。
    thread: ChatHistoryAgentThread | None = None

    # 定义用户输入的问题列表（这里模拟了一个去墨尔本的家庭旅行需求）
    user_inputs = [
        "为一个有两个孩子的四口之家制定一个从新加坡到墨尔本的旅行计划",
    ]

    for user_input in user_inputs:
        
        # --- 1. 构建 UI 容器：显示用户提问 ---
        html_output = "<div style='margin-bottom:10px'>"
        html_output += "<div style='font-weight:bold'>User:</div>"
        html_output += f"<div style='margin-left:20px'>{user_input}</div>"
        html_output += "</div>"

        # --- 2. 核心调用：获取 Agent 响应 ---
        # 使用 await 异步等待 Agent 的回复。get_response 会处理提示词并返回结构化数据。
        response = await agent.get_response(messages=user_input, thread=thread)
        # 更新 thread 变量，以便在连续对话中保持上下文同步
        thread = response.thread

        try:
            # --- 3. 数据校验 (Validation) ---
            # 关键步骤：使用 json.loads 解析字符串，然后喂给 Pydantic 模型进行校验。
            # 如果模型返回的 JSON 字段缺失或类型不对，这里会抛出异常。
            travel_plan = TravelPlan.model_validate(json.loads(response.message.content))

            # 如果校验通过，将 Pydantic 对象转换回带缩进的漂亮 JSON 字符串
            formatted_json = travel_plan.model_dump_json(indent=4)
            
            # 构建 UI 容器：显示校验成功后的结构化计划
            html_output += "<div style='margin-bottom:20px'>"
            html_output += "<div style='font-weight:bold'>Validated Travel Plan:</div>"
            # 使用 <pre> 标签保留 JSON 的换行和空格格式
            html_output += f"<pre style='margin-left:20px; padding:10px; border-radius:5px;'>{formatted_json}</pre>"
            html_output += "</div>"
            
        except ValidationError as e:
            # --- 4. 错误处理 ---
            # 如果大模型“翻车”（没按格式输出），捕获 Pydantic 校验错误并标红显示
            html_output += "<div style='margin-bottom:20px; color:red;'>"
            html_output += "<div style='font-weight:bold'>Validation Error:</div>"
            html_output += f"<pre style='margin-left:20px;'>{str(e)}</pre>"
            html_output += "</div>"
            
            # 为了方便调试，即便报错也展示出原始的回复内容（Raw Response）
            html_output += "<div style='margin-bottom:20px;'>"
            html_output += "<div style='font-weight:bold'>Raw Response:</div>"
            html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.content}</div>"
            html_output += "</div>"

        # 在每轮对话结果后添加一条水平分割线
        html_output += "<hr>"

        # 最终调用 display，将拼接好的 HTML 渲染到页面上
        display(HTML(html_output))

# 启动异步主程序
await main()

好的，请提供需要翻译的 Markdown 文件内容，我将根据规则进行翻译。



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
